# 📝 Ruchith Reddy Parnem — Text Analyst
## Multimodal Crime / Incident Report Analyzer

**Data Type:** Social media posts / news articles / crime text reports  
**Objective:** NLP analysis → NER, sentiment, topic classification → structured CSV

### Pipeline:
1. Load and clean text data
2. Named Entity Recognition (spaCy)
3. Sentiment analysis (HuggingFace)
4. Topic classification (zero-shot)
5. Output structured CSV


In [2]:
# =============================================
# CELL 1: Install Dependencies
# =============================================
!pip install -q spacy transformers pandas torch nltk
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 87.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# =============================================
# CELL 2: Import Libraries
# =============================================
import spacy
import pandas as pd
import re
import nltk
import warnings
warnings.filterwarnings('ignore')

from transformers import pipeline
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Load models
print("Loading spaCy NER model...")
nlp = spacy.load("en_core_web_sm")

print("Loading sentiment analysis model...")
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

print("Loading zero-shot topic classifier...")
topic_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

print("All models loaded!")


Loading spaCy NER model...
Loading sentiment analysis model...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

All models loaded!


#### Dataset: CrimeReport from Kaggle: https://www.kaggle.com/datasets/cameliasiadat/crimereport



In [11]:
# =============================================
# CELL 3: Load CrimeReport Dataset
# =============================================
import os, glob, json
os.makedirs("outputs", exist_ok=True)

txt_file = glob.glob("/content/CrimeReport (1).txt")[0]

# This dataset is JSON tweets — one per line
records = []
with open(txt_file, 'r') as f:
    for line in f:
        try:
            tweet = json.loads(line.strip())
            text = tweet.get("text", "")
            place = tweet.get("place", {})
            location = place.get("full_name", "Unknown") if place else "Unknown"
            created = tweet.get("created_at", "")
            records.append({
                "Text_ID": f"TXT_{len(records)+1:03d}",
                "Source": "Twitter",
                "Raw_Text": text,
                "Location_Raw": location,
                "Date": created
            })
        except:
            continue

df_raw = pd.DataFrame(records)
print(f"Loaded: {len(df_raw)} crime tweets")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

Loaded: 115 crime tweets
Columns: ['Text_ID', 'Source', 'Raw_Text', 'Location_Raw', 'Date']


,Text_ID,Source,Raw_Text,Location_Raw,Date
0,TXT_001,Twitter,Active crime scene on I-59/20 near Jeff/Tusc C...,"Hoover, AL",Fri Jan 31 05:51:59 +0000 2014
1,TXT_002,Twitter,Police have arrested a suspect in the Monday s...,Unknown,Fri Jan 31 05:01:39 +0000 2014
2,TXT_003,Twitter,Lawsuit alleges Chicago Police strip-searched ...,Unknown,Fri Jan 31 18:10:37 +0000 2014
3,TXT_004,Twitter,New NRA Advice: Don't Cooperate With Police If...,Unknown,Fri Jan 31 02:33:17 +0000 2014
4,TXT_005,Twitter,"Police failing to stamp out ‘honour crimes’, f...",Unknown,Thu Jan 30 23:18:45 +0000 2014


In [13]:
# =============================================
# CELL 4: Text Preprocessing
# =============================================
import nltk
import re
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'-]', '', text)
    text = ' '.join(text.split())
    return text.strip()

def tokenize_text(text):
    tokens = word_tokenize(text.lower())
    filtered = [t for t in tokens if t.isalpha() and t not in stop_words]
    return filtered

text_col = "Raw_Text"
print(f"Using text column: {text_col}")

df_raw["Clean_Text"] = df_raw[text_col].astype(str).apply(clean_text)
df_raw["Tokens"] = df_raw["Clean_Text"].apply(tokenize_text)
df_raw["Token_Count"] = df_raw["Tokens"].apply(len)

print(f"Avg tokens per record: {df_raw['Token_Count'].mean():.1f}")
print(f"\nSample cleaned text:")
print(df_raw["Clean_Text"].iloc[0][:150])

Using text column: Raw_Text
Avg tokens per record: 10.9

Sample cleaned text:
Active crime scene on I-5920 near JeffTusc Co line. One dead, one injured shooting involved. Police search in the area traffic stopped


In [14]:
# =============================================
# CELL 5: Named Entity Recognition (NER) with spaCy
# =============================================

def extract_entities(text):
    """Extract named entities from text."""
    doc = nlp(text)
    entities = {
        "PERSON": [],
        "GPE": [],      # Countries, cities, states
        "LOC": [],      # Non-GPE locations
        "FAC": [],      # Facilities, buildings, streets
        "ORG": [],      # Organizations
        "DATE": [],
        "TIME": [],
    }
    for ent in doc.ents:
        if ent.label_ in entities:
            entities[ent.label_].append(ent.text)

    return entities

def get_location_string(entities):
    """Combine location entities into a string."""
    locations = entities.get("GPE", []) + entities.get("LOC", []) + entities.get("FAC", [])
    return ", ".join(list(set(locations))) if locations else "Unknown"

def get_entities_string(entities):
    """Format all entities into a readable string."""
    parts = []
    for label, values in entities.items():
        if values:
            unique_vals = list(set(values))
            parts.append(f"{label}: {', '.join(unique_vals)}")
    return "; ".join(parts) if parts else "None"

# Extract entities
print("Extracting named entities...")
df_raw["Entities_Raw"] = df_raw["Clean_Text"].apply(extract_entities)
df_raw["Location_Entity"] = df_raw["Entities_Raw"].apply(get_location_string)
df_raw["Entities"] = df_raw["Entities_Raw"].apply(get_entities_string)

print("\nSample entities extracted:")
for _, row in df_raw.head(3).iterrows():
    print(f"  Text: {row['Clean_Text'][:60]}...")
    print(f"  Location: {row['Location_Entity']}")
    print(f"  Entities: {row['Entities']}")
    print()


Extracting named entities...

Sample entities extracted:
  Text: Active crime scene on I-5920 near JeffTusc Co line. One dead...
  Location: Unknown
  Entities: None

  Text: Police have arrested a suspect in the Monday shooting at the...
  Location: Unknown
  Entities: ORG: Word Tabernacle Church; DATE: Monday

  Text: Lawsuit alleges Chicago Police strip-searched trio in public...
  Location: Chicago
  Entities: GPE: Chicago



In [15]:
# =============================================
# CELL 6: Sentiment Analysis
# =============================================

def analyze_sentiment(text):
    """Analyze sentiment of text."""
    try:
        result = sentiment_analyzer(text[:512])[0]
        return result["label"], round(result["score"], 3)
    except:
        return "UNKNOWN", 0.0

# Run sentiment analysis
print("Running sentiment analysis...")
sentiments = df_raw["Clean_Text"].apply(analyze_sentiment)
df_raw["Sentiment"] = sentiments.apply(lambda x: x[0])
df_raw["Sentiment_Score"] = sentiments.apply(lambda x: x[1])

print("Sentiment distribution:")
print(df_raw["Sentiment"].value_counts())


Running sentiment analysis...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Sentiment distribution:
Sentiment
NEGATIVE    105
POSITIVE     10
Name: count, dtype: int64


In [16]:
# =============================================
# CELL 7: Topic / Crime Type Classification
# =============================================

CRIME_LABELS = [
    "robbery / theft",
    "fire / arson",
    "vehicle accident",
    "assault / violence",
    "burglary / break-in",
    "shooting / gun violence",
    "public disturbance",
    "suspicious activity",
    "drug-related crime",
    "vandalism"
]

def classify_topic(text):
    """Zero-shot crime topic classification."""
    try:
        result = topic_classifier(text[:512], CRIME_LABELS, multi_label=False)
        top_label = result["labels"][0]
        top_score = round(result["scores"][0], 3)
        return top_label, top_score
    except Exception as e:
        return "unknown", 0.0

# Run topic classification (this may take a few minutes)
print("Running zero-shot topic classification...")
print("(This uses BART-large-MNLI and may take 1-2 minutes)\n")

topics = []
for idx, row in df_raw.iterrows():
    topic, score = classify_topic(row["Clean_Text"])
    topics.append((topic, score))
    print(f"  [{idx+1}/{len(df_raw)}] {topic} ({score:.3f})")

df_raw["Topic"] = [t[0] for t in topics]
df_raw["Topic_Score"] = [t[1] for t in topics]

print("\nTopic distribution:")
print(df_raw["Topic"].value_counts())


Running zero-shot topic classification...
(This uses BART-large-MNLI and may take 1-2 minutes)

  [1/115] shooting / gun violence (0.603)
  [2/115] shooting / gun violence (0.642)
  [3/115] suspicious activity (0.606)
  [4/115] suspicious activity (0.527)
  [5/115] suspicious activity (0.538)
  [6/115] suspicious activity (0.614)
  [7/115] robbery / theft (0.666)
  [8/115] assault / violence (0.482)
  [9/115] suspicious activity (0.519)
  [10/115] drug-related crime (0.946)
  [11/115] suspicious activity (0.420)
  [12/115] suspicious activity (0.608)
  [13/115] suspicious activity (0.352)
  [14/115] suspicious activity (0.399)
  [15/115] suspicious activity (0.679)
  [16/115] suspicious activity (0.625)
  [17/115] suspicious activity (0.571)
  [18/115] suspicious activity (0.571)
  [19/115] suspicious activity (0.571)
  [20/115] suspicious activity (0.377)
  [21/115] suspicious activity (0.571)
  [22/115] suspicious activity (0.377)
  [23/115] suspicious activity (0.563)
  [24/115] sus

In [17]:
# =============================================
# CELL 8: Severity Classification
# =============================================

def classify_severity(text, topic, sentiment):
    """Classify incident severity based on multiple signals."""
    text_lower = text.lower()

    high_indicators = ["shooting", "gun", "killed", "dead", "fatal", "trapped",
                       "serious injuries", "knife", "stabbed", "explosion", "bomb"]
    medium_indicators = ["injured", "accident", "fire", "assault", "fight",
                         "arrested", "hospital", "damage", "evacuated"]
    low_indicators = ["suspicious", "noise", "minor", "packages", "vandalism"]

    score = 0
    for word in high_indicators:
        if word in text_lower:
            score += 3
    for word in medium_indicators:
        if word in text_lower:
            score += 2
    for word in low_indicators:
        if word in text_lower:
            score += 1

    # Boost by topic
    high_topics = ["shooting / gun violence", "assault / violence", "fire / arson"]
    medium_topics = ["robbery / theft", "vehicle accident", "burglary / break-in"]
    if topic in high_topics:
        score += 2
    elif topic in medium_topics:
        score += 1

    if score >= 5:
        return "High"
    elif score >= 2:
        return "Medium"
    else:
        return "Low"

df_raw["Severity_Label"] = df_raw.apply(
    lambda row: classify_severity(row["Clean_Text"], row["Topic"], row["Sentiment"]),
    axis=1
)

print("Severity distribution:")
print(df_raw["Severity_Label"].value_counts())


Severity distribution:
Severity_Label
Low       79
Medium    22
High      14
Name: count, dtype: int64


In [18]:
# =============================================
# CELL 9: Generate Final Structured CSV
# =============================================

# Build output DataFrame
if "Text_ID" not in df_raw.columns:
    df_raw["Text_ID"] = [f"TXT_{i:03d}" for i in range(1, len(df_raw)+1)]
if "Source" not in df_raw.columns:
    df_raw["Source"] = "Text Report"

# Format crime type
df_raw["Crime_Type"] = df_raw["Topic"].str.title()

output_cols = ["Text_ID", "Crime_Type", "Location_Entity", "Sentiment", "Topic", "Severity_Label"]
df_text_output = df_raw[output_cols].copy()

df_text_output.to_csv("outputs/text_analyst_output.csv", index=False)

print("=" * 70)
print("TEXT ANALYST - FINAL OUTPUT")
print("=" * 70)
print(f"Total texts processed: {len(df_text_output)}")
print(f"Output saved to: outputs/text_analyst_output.csv")
print("=" * 70)
df_text_output


TEXT ANALYST - FINAL OUTPUT
Total texts processed: 115
Output saved to: outputs/text_analyst_output.csv


,Text_ID,Crime_Type,Location_Entity,Sentiment,Topic,Severity_Label
0,TXT_001,Shooting / Gun Violence,Unknown,NEGATIVE,shooting / gun violence,High
1,TXT_002,Shooting / Gun Violence,Unknown,NEGATIVE,shooting / gun violence,High
2,TXT_003,Suspicious Activity,Chicago,NEGATIVE,suspicious activity,Low
3,TXT_004,Suspicious Activity,Unknown,NEGATIVE,suspicious activity,Medium
4,TXT_005,Suspicious Activity,Unknown,NEGATIVE,suspicious activity,Low
...,...,...,...,...,...,...
110,TXT_111,Suspicious Activity,Unknown,NEGATIVE,suspicious activity,Low
111,TXT_112,Shooting / Gun Violence,Unknown,NEGATIVE,shooting / gun violence,High
112,TXT_113,Suspicious Activity,Unknown,NEGATIVE,suspicious activity,Low
113,TXT_114,Suspicious Activity,Unknown,NEGATIVE,suspicious activity,Low


### ✅ Text Analyst Complete!
**Output:** `outputs/text_analyst_output.csv`
